# RPS PreTrained YOLO Fine-tuning

## g-drive 마운트

In [ ]:
try:
    from google.colab import drive
    drive.mount('/content/drive')
    print('g-drive mounted.')
    colab=True
except:
    print('local drive.')
    colab =False

Mounted at /content/drive
g-drive mounted.


## 경로 설정

In [ ]:
# 다른 경로에 저장하려면, 아래 경로를 수정하세요
if colab :
  save_dir = '/content/drive/MyDrive/files/save/'
  dataset_zip = '/content/drive/MyDrive/files/RPS_Dataset_YOLO.zip'
  dataset_root = '/content/RPS_Dataset_YOLO'
  project_root = '/content/rps_runs'
else :
  save_dir = '../files/save/'
  dataset_zip = '../files/RPS_Dataset_YOLO.zip'
  dataset_root = './RPS_Dataset_YOLO'
  project_root = './rps_runs'

## RPS 데이터셋 준비

In [ ]:
!unzip -q -o {dataset_zip}

## Ultralytics 설치 및 import

In [ ]:
!pip -q install -U ultralytics==8.4.98

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 42.1/42.1 kB 2.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.4/1.4 MB 30.8 MB/s eta 0:00:00


## 학습

In [ ]:
DATA = f'{dataset_root}/data.yaml'

!yolo detect train model=yolo11n.pt data={DATA} \
  imgsz=320 epochs=80 batch=32 device=0 workers=4 \
  project={project_root} degrees=15.0 flipud=0.5 \
  fliplr=0.5 mosaic=1.0 name=rps_yolo11n

Creating new Ultralytics Settings v0.0.6 file ✅ 
View Ultralytics Settings with 'yolo settings' or at '/root/.config/Ultralytics/settings.json'
Update Settings with 'yolo settings key=value', i.e. 'yolo settings runs_dir=path/to/dir'. For help see https://docs.ultralytics.com/quickstart/#ultralytics-settings.
Ultralytics 8.4.98 🚀 Python-3.12.13 torch-2.11.0+cu128 CUDA:0 (Tesla T4, 14913MiB)
engine/trainer: agnostic_nms=False, amp=True, angle=1.0, augment=False, auto_augment=randaugment, batch=32, bgr=0.0, box=7.5, cache=False, cfg=None, classes=None, close_mosaic=10, cls=0.5, cls_pw=0.0, cls_remap=True, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=/content/RPS_Dataset_YOLO/data.yaml, degrees=15.0, deterministic=True, device=0, dfl=1.5, dis=6.0, distill_model=None, dnn=False, dropout=0.0, dynamic=False, embed=None, end2end=None, epochs=80, erasing=0.4, exist_ok=False, fliplr=0.5, flipud=0.5, format=torchscript, fraction=1.0, freeze=None,

## LiteRT로 변환

In [ ]:
BEST = f'{project_root}/rps_yolo11n/weights/best.pt'

# 양자화 안함
!yolo export model={BEST} format=litert imgsz=320 data={DATA}

Ultralytics 8.4.98 🚀 Python-3.12.13 torch-2.11.0+cu128 CPU (Intel Xeon CPU @ 2.00GHz)
💡 ProTip: Export to OpenVINO format for best performance on Intel hardware. Learn more at https://docs.ultralytics.com/integrations/openvino/
YOLO11n summary (fused): 101 layers, 2,582,737 parameters, 0 gradients, 6.3 GFLOPs

PyTorch: starting from '/content/rps_runs/rps_yolo11n/weights/best.pt' with input shape (1, 3, 320, 320) BCHW and output shape(s) (1, 7, 2100) (5.2 MB)
requirements: Ultralytics requirements ['litert-torch>=0.9.0', 'ai-edge-litert>=2.1.4'] not found, attempting AutoUpdate...
Using Python 3.12.13 environment at: /usr
Resolved 85 packages in 573ms
Prepared 15 packages in 3.34s
Uninstalled 3 packages in 37ms
Installed 15 packages in 38ms
 + ai-edge-litert==2.1.5
 + ai-edge-quantizer==0.7.0
 + backports-strenum==1.3.1
 + fire==0.7.1
 - immutabledict==4.3.1
 + immutabledict==4.2.1
 + jaxtyping==0.3.11
 + litert-converter==0.2.0
 + litert-lm-builder==0.14.0
 + litert-torch==0.9.1
 + or

In [ ]:
# PTQ(INT8) (입력과 출력은 float32)
!yolo export model={BEST} format=litert imgsz=320 data={DATA} quantize=8

Ultralytics 8.4.98 🚀 Python-3.12.13 torch-2.11.0+cu128 CPU (Intel Xeon CPU @ 2.00GHz)
WARNING ⚠️ LiteRT INT8 export does not support end2end models, disabling end2end branch.
YOLO11n summary (fused): 101 layers, 2,582,737 parameters, 0 gradients, 6.3 GFLOPs

PyTorch: starting from '/content/rps_runs/rps_yolo11n/weights/best.pt' with input shape (1, 3, 320, 320) BCHW and output shape(s) (1, 7, 2100) (5.2 MB)
LiteRT: collecting INT8 calibration images from 'data=/content/RPS_Dataset_YOLO/data.yaml'
val: Fast image access ✅ (ping: 0.0±0.0 ms, read: 2157.6±452.5 MB/s, size: 70.4 KB)
val: Scanning /content/RPS_Dataset_YOLO/labels/test.cache... 22 images, 0 backgrounds, 0 corrupt: 100% ━━━━━━━━━━━━ 22/22 2.9Mit/s 0.0s
WARNING ⚠️ LiteRT: >300 images recommended for INT8 calibration, found 22 images.
Failed to load /usr/local/lib/python3.12/dist-packages/torchao/_C_mxfp8.cpython-310-x86_64-linux-gnu.so: Could not load this library: /usr/local/lib/python3.12/dist-packages/torchao/_C_mxfp8.cpyth

In [ ]:
# PTQ(Dynamic Range)
!yolo export model={BEST} format=litert imgsz=320 data={DATA} quantize=w8a32

Ultralytics 8.4.98 🚀 Python-3.12.13 torch-2.11.0+cu128 CPU (Intel Xeon CPU @ 2.00GHz)
YOLO11n summary (fused): 101 layers, 2,582,737 parameters, 0 gradients, 6.3 GFLOPs

PyTorch: starting from '/content/rps_runs/rps_yolo11n/weights/best.pt' with input shape (1, 3, 320, 320) BCHW and output shape(s) (1, 7, 2100) (5.2 MB)
Failed to load /usr/local/lib/python3.12/dist-packages/torchao/_C_mxfp8.cpython-310-x86_64-linux-gnu.so: Could not load this library: /usr/local/lib/python3.12/dist-packages/torchao/_C_mxfp8.cpython-310-x86_64-linux-gnu.so
Failed to load /usr/local/lib/python3.12/dist-packages/torchao/_C_cutlass_90a.abi3.so: Could not load this library: /usr/local/lib/python3.12/dist-packages/torchao/_C_cutlass_90a.abi3.so

LiteRT: starting export with litert_torch 0.9.1...
(00:00) [START] LiteRT-Torch Convert
(00:00) [START] LiteRT-Torch Convert > Torch Export: serving_default
(00:01) [START] LiteRT-Torch Convert > Torch Export: serving_default > 
ExportedProgram Run Decompositions
/us

## tflite 파일 저장

In [ ]:
!cp {project_root}/rps_yolo11n/weights/*.tflite {save_dir}